# TTM × SIX G1 — Qwen3-1.7B readiness, no science

Dùng Colab Free Tesla T4. Chỉ kiểm **mô hình đích** 1.7B, không chạy lại Qwen3-0.6B. **Ô 1:** xác minh mã nguồn Git blob, biên dịch và tiền kiểm; không tải trọng số. **Ô 2:** chỉ sau ô 1 PASS, chủ sở hữu chủ động chạy đúng một lần để tải ~4.08 GB từ revision đã ghim, kiểm SHA-256 hai shard trước khi load và kiểm 29 hidden states trên một chuỗi tổng hợp. Không có TRAIN/DEV/TEST, không có khoa học, không ghi Drive.

Nếu ô 1 FAIL, không chạy ô 2. Nếu ô 2 FAIL, giữ nguyên receipt/marker; không chạy lại để tìm kết quả. Chỉ sử dụng kết quả để quyết định khả thi phần cứng G1, không tự mở science lock.


In [ ]:
# G1 target17 — one integrated zero-science environment preflight (no model weights)
import ast, hashlib, importlib.metadata as md, json, pathlib, platform, py_compile, subprocess, sys, urllib.request
ROOT=pathlib.Path("/content/ttm_six_g1_target17")
SCRIPT_BLOB="1a815a5572136a93cd9a292705b75a8a17002a97"
SCRIPT_COMMIT="29907814c58cb3e78355c0d84497a8f8952d0437"
SCRIPT_PATH="scripts/research/six_ttm/g1_qwen17_zero_science_technical_readiness_v1.py"
URL="https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"+SCRIPT_COMMIT+"/"+SCRIPT_PATH
print("G1_TARGET17_NOTEBOOK_PYTHON="+platform.python_version())
print("G1_TARGET17_NOTEBOOK_PYTHON_EXE="+sys.executable)
data=urllib.request.urlopen(URL,timeout=45).read()
blob=hashlib.sha1(b"blob "+str(len(data)).encode("ascii")+b"\0"+data).hexdigest()
print("G1_TARGET17_SOURCE_BLOB="+blob)
if blob!=SCRIPT_BLOB: raise RuntimeError("STOP_SCRIPT_PIN_MISMATCH")
ast.parse(data.decode("utf-8"),filename=SCRIPT_PATH)
if not pathlib.Path("/content").is_dir(): raise RuntimeError("STOP_NOT_COLAB")
ROOT.mkdir(parents=True,exist_ok=True)
script=ROOT/"g1_qwen17_zero_science_technical_readiness_v1.py"
script.write_bytes(data)
py_compile.compile(str(script),doraise=True)
print("G1_TARGET17_STATIC_COMPILE=PASS")
proc=subprocess.run([sys.executable,str(script),"--phase","preflight"],check=False)
print("G1_TARGET17_PREFLIGHT_EXIT="+str(proc.returncode))
receipt=ROOT/"G1_TARGET17_PREFLIGHT_RECEIPT.json"
if receipt.is_file():
    print("G1_TARGET17_PREFLIGHT_RECEIPT="+receipt.read_text("utf-8"))
else:
    print("G1_TARGET17_PREFLIGHT_RECEIPT=MISSING")
if proc.returncode:
    print("G1_TARGET17_NEXT=STOP_NO_MODEL_DOWNLOAD")
else:
    print("G1_TARGET17_NEXT=PASS_PREFLIGHT_ONLY")


In [ ]:
# G1 target17 — run once only AFTER preflight PASS; synthetic data only.
import hashlib, importlib.metadata as md, json, pathlib, platform, subprocess, sys
ROOT=pathlib.Path("/content/ttm_six_g1_target17")
script=ROOT/"g1_qwen17_zero_science_technical_readiness_v1.py"
expected_blob="1a815a5572136a93cd9a292705b75a8a17002a97"
if not script.is_file(): raise RuntimeError("STOP_SCRIPT_NOT_FOUND")
data=script.read_bytes()
actual_blob=hashlib.sha1(b"blob "+str(len(data)).encode("ascii")+b"\0"+data).hexdigest()
if actual_blob!=expected_blob: raise RuntimeError("STOP_SOURCE_CHANGED")
receipt_path=ROOT/"G1_TARGET17_PREFLIGHT_RECEIPT.json"
if not receipt_path.is_file(): raise RuntimeError("STOP_PREFLIGHT_RECEIPT_MISSING")
receipt=json.loads(receipt_path.read_text("utf-8"))
if receipt.get("status")!="PASS_PREFLIGHT_ONLY":
    raise RuntimeError("STOP_PREFLIGHT_NOT_PASS")
env=receipt.get("environment",{})
if env.get("python")!=platform.python_version() or env.get("python_executable")!=sys.executable:
    raise RuntimeError("STOP_PYTHON_CHANGED")
expect={"transformers":"5.22.1","huggingface-hub":"1.33.0",
        "tokenizers":"0.23.3","safetensors":"0.8.0"}
if env.get("packages")!=expect: raise RuntimeError("STOP_ENV_PACKAGES_NOT_PINNED")
for name,version in expect.items():
    if md.version(name)!=version: raise RuntimeError("STOP_ENV_PACKAGE_CHANGED_"+name)
if md.version("torch")!=env.get("torch"): raise RuntimeError("STOP_TORCH_CHANGED")
if env.get("gpu_name")!="Tesla T4": raise RuntimeError("STOP_NOT_T4")
marker=ROOT/"G1_TARGET17_ONE_SHOT_CONSUMED.marker"
with marker.open("x",encoding="utf-8") as s:
    s.write("CONSUMED_BEFORE_MODEL_DOWNLOAD\n")
try:
    proc=subprocess.run([sys.executable,str(script),"--phase","smoke"],check=False)
    print("G1_TARGET17_SMOKE_EXIT="+str(proc.returncode))
    receipt=ROOT/"G1_TARGET17_SMOKE_RECEIPT.json"
    print("G1_TARGET17_SMOKE_RECEIPT="+
          (receipt.read_text("utf-8") if receipt.is_file() else "MISSING"))
finally:
    print("G1_TARGET17_ONE_SHOT=CONSUMED_DO_NOT_RETRY")
